# CPGF step-edge example

This is a short example for trying one filter. It makes a `1024 x 1024` image where the left half is black and the right half is white. Then it runs the Circular Polynomial Gradient Filter (CPGF) and plots the two outputs.

Change `radius` and `degree`, rerun the notebook, and compare the outputs. Those two values are the main CPGF settings to experiment with.


First import only what this example needs. `time` is only here so the notebook can print how long the filter took to run.


In [ ]:
import time

import matplotlib.pyplot as plt
import torch

from agfb_filters import ExecutionPath, cpgf_definition, run_filter

Set the CPGF parameters here. A larger `radius` uses a bigger neighborhood around each pixel. A larger `degree` fits a higher-order polynomial. Some combinations may not be valid because there are not enough pixels in the support to fit that polynomial.

`ExecutionPath.SPARSE_OFFSETS` is a good default for CPGF. You can also try `ExecutionPath.SPATIAL_DENSE` to run the same filter with a dense-kernel path.


In [ ]:
radius = 2
degree = 2
path = ExecutionPath.SPARSE_OFFSETS

size = 1024
image = torch.zeros(1, size, size)
image[:, :, size // 2 :] = 1.0

definition = cpgf_definition(radius=radius, degree=degree)

start = time.perf_counter()
gradient_x, gradient_y = run_filter(
    definition,
    image,
    path=path,
    boundary=definition.default_boundary,
)
elapsed = time.perf_counter() - start

print(f"{elapsed:.4f} seconds")

The image changes left-to-right, so the main response should appear in `gradient_x`. The `gradient_y` output should be mostly blank because the image does not change top-to-bottom.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

axes[0].imshow(gradient_x[0], cmap="gray")
axes[0].set_title("gradient_x")

axes[1].imshow(gradient_y[0], cmap="gray")
axes[1].set_title("gradient_y")

for ax in axes:
    ax.axis("off")

fig.suptitle(f"CPGF radius={radius}, degree={degree}, time={elapsed:.4f} s")
fig.tight_layout()